In [1]:
import pandas as pd
import numpy as np
import requests
import xmltodict
import json
import traceback
from pandas.tseries.offsets import MonthEnd
from DATA.stock_invest_function import *
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
from sqlalchemy import create_engine
import pymysql
from tqdm import tqdm
from DATA.stock_invest_function import *
import socket

In [2]:
# db_info = {
#     'host' : get_db_host(),
#     'port' : 3307,
#     'user' : 'stox7412',
#     'password' : 'Apt106503!~',
#     'database' : 'investar'
# }

# kr_path = r'C:\Users\MetaM\PycharmProjects\pythonProject3\HS_Code_500\HS_Code_500.xlsx'
# us_path = r'C:\Users\MetaM\PycharmProjects\stock_forecast\DATA\미국_200대_수출금액_.HScode_202506.xlsx'
#
# kr_data = pd.read_excel(kr_path)
# us_data = pd.read_excel(us_path)
#
# kr_data['country'] = 'KR'
# us_data['country'] = 'US'
#
# company_df = fetch_table_data(db_info, 'hs_code_by_kr_monster_company')
#
# hscode_df = pd.concat([kr_data, us_data])
# hscode_df['hs_code_6d'] = hscode_df ['HS_Code'].astype(str).str[:6]
# hscode_df_resize = hscode_df.drop_duplicates(subset=['hs_code_6d', 'country'], keep='first')

In [3]:
path1 = r"C:\Users\MetaM\Downloads\toptier_company_final_ver_20250814.xlsx"
path2 = r"C:\Users\MetaM\Downloads\hscode_info_block1.xlsx"

df1 = pd.read_excel(path1)
df2 = pd.read_excel(path2)

In [11]:
hscode_data = pd.concat([df1, df2], join='inner')

# Code, Name, hs_code가 모두 같은 행에서 첫 번째만 남기고 제거
df_unique = hscode_data.drop_duplicates(subset=['Code', 'Name', 'hs_code'], keep='first')

hs_code_df = df_unique['hs_code']

In [15]:
df_unique

,Code,Name,hs_code
0,A000080,하이트진로,2208904000
1,A000210,DL,39022
2,A000210,DL,3901101000
3,A000660,SK하이닉스,8542321010
4,A000660,SK하이닉스,8542321030
...,...,...,...
446,A020000,한섬,6103
447,A005390,신성통상,6202
448,A008290,원풍물산,6203
449,A105630,한세실업,6102


In [13]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',           # 예: 'root'
    'password': 'Apt106503!~',   # 예: '1234'
    'host': get_db_host(),           # 예: 'localhost'
    'port': '3307',                # 예: '3306'
    'database': 'investar'    # 예: 'trade_data'

}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 테이블 이름 지정
table_name = 'target_hs_code'

# DB에 업로드 (기존 테이블 덮어씀 → 'replace', 추가는 'append')
hs_code_df.to_sql(name=table_name, con=engine, index=False, if_exists='replace')

print(f"✅ {table_name} 테이블에 {len(hs_code_df)}개의 행이 업로드되었습니다.")

✅ target_hs_code 테이블에 808개의 행이 업로드되었습니다.


In [14]:
df_unique

,Code,Name,hs_code
0,A000080,하이트진로,2208904000
1,A000210,DL,39022
2,A000210,DL,3901101000
3,A000660,SK하이닉스,8542321010
4,A000660,SK하이닉스,8542321030
...,...,...,...
446,A020000,한섬,6103
447,A005390,신성통상,6202
448,A008290,원풍물산,6203
449,A105630,한세실업,6102


In [16]:
# SQLAlchemy 연결 URL 생성
db_url = f"mysql+pymysql://{db_info['user']}:{db_info['password']}@" \
         f"{db_info['host']}:{db_info['port']}/{db_info['database']}"

# DB 엔진 생성
engine = create_engine(db_url)

# 테이블 저장
df_unique.to_sql(
    name="target_company_db",   # 테이블명
    con=engine,
    if_exists="replace",        # 기존 테이블 있으면 덮어쓰기, append 로 변경 가능
    index=False                 # DataFrame index는 저장 안 함
)

print("데이터가 DB에 저장되었습니다.")

데이터가 DB에 저장되었습니다.
